# 02_Step1VsMonster_allvisits: step-1 photometry against the Monster, all visits and bands of the run

Same method as `01_Step1VsMonster` (pipeline reference selection, 1 arcsec match, `psfFlux` mag minus Monster ComCam mag), applied to every visit of the stage-1 collection.

Outputs (in `data_02_Step1VsMonster_allvisits/`):
- `matches_all.csv`: every matched star (visit, band, detector, magnitudes, colours, flag, dmag);
- `summary_per_detector.csv`: median and sigma_MAD of dmag per (visit, band, detector);
- `summary_colour_term.csv`: linear fit dmag = slope * (g - i) + offset per (visit, band) for bright stars.

Questions: is the colour slope seen in visit 2025041700536 (i band, 0.0086 mag per mag of g-i) present in the other bands and visits? Does it differ between ITL and E2V sensors? Are there detector-to-detector or visit-to-visit offsets?

Run with the `lsst` kernel at CC-IN2P3.

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-03
- last update : 2026-11-03 : use input collection : `u/dagoret/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage1`
- last update : 2026-11-05 : use input collection : `u/dagoret/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1`
- `DP2 tutorial` : https://dp2.lsst.io/tutorials/notebook/204/notebook-204-2.html
- `monster catalog documentation` : https://dmtn-277.lsst.io/DMTN-277.pdf
- `quantum graph for step1a` : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_step1a-single-visit-detectors.pdf

In [ ]:
import os
import sys
from pathlib import Path
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
import astropy.units as u

from IPython.display import display, Image, Markdown

SRC = Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rubincc import monster as mon

NB_TAG = '02_Step1VsMonster_allvisits'
DATA_DIR = Path(f'data_{NB_TAG}')
FIG_DIR = Path(f'figs_{NB_TAG}')
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.labelsize'] = 'x-large'
plt.rcParams['axes.titlesize'] = 'x-large'
plt.rcParams['xtick.labelsize'] = 'x-large'
plt.rcParams['ytick.labelsize'] = 'x-large'


def savefig(fig, name):
    # every figure is saved as PDF and PNG
    for ext in ('pdf', 'png'):
        fig.savefig(FIG_DIR / f'{NB_TAG}_{name}.{ext}', bbox_inches='tight', dpi=150)


def robust_stats(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    return med, 1.4826 * np.median(np.abs(x - med))

In [ ]:
# Configure logging for Rubin Science Pipelines
logging.basicConfig(
    level=logging.INFO,
    format="%(name)s %(levelname)s: %(message)s",
    force=True,
)

# Enable INFO messages from LSST packages
logging.getLogger("lsst").setLevel(logging.INFO)

In [ ]:
def ShowOneImage(filename_img, title = None, caption = None):
    """
    """
    print(os.path.abspath(filename_img))
    print(os.path.exists(filename_img))
    with open(filename_img, "rb") as f:
        data = f.read()
    print("File size:", len(data), "bytes")
    print("PNG signature:", data[:8])
    if title is not None:
        display(Markdown(f"### {title}"))
    display(Image(data=data))
    if caption is not None:
        display(Markdown(f"### {caption}"))

In [ ]:
REPO = 'main'
USER = os.getenv('USER')
#RUN_COLLECTION = f'u/{USER}/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage1'
RUN_COLLECTION = f'u/{USER}/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1'

MONSTER_DATASET = mon.DEFAULT_MONSTER_DATASET
MONSTER_COLLECTIONS = 'refcats/DM-49042/the_monster_20250219'
REF = 'monster_ComCam'

MAX_VISITS = None       # None = all visits of the selection below
SAMPLE_PER_BAND = None    # visits per band, spread over the run (None = all visits)
MAX_SEP_ARCSEC = 0.5    # 99% of the matches of visit 2025041700536 are below 0.47 arcsec
MAG_RANGE = (16.0, 20.0)
BRIGHT_MAX = 18.5       # bright stars used for the colour term
MIN_STARS = 15          # minimum number of stars per (visit, band, detector) in the summaries
LIMIT = 10000

In [ ]:
# figure with quantum diagram
filename_diagram = "../fig_diagrams/pipeline_drp_pipe_LSSTCam_DRP_step1a-single-visit-detectors.png"
ShowOneImage(filename_diagram,title="DRP step1a/detectors")

## 1. Visits of the run

- Visit are taken from butler registry
- I hope those visits are really filtered as science exposures
- There is no selection on the observation_type and observation_reason
- because the TARGET_NAME is COMOS, it shows that the visits are the one I have selected

In [ ]:
run_butler = mon.get_butler(REPO, collections=RUN_COLLECTION)
refs = run_butler.query_datasets('single_visit_star', limit=LIMIT)
all_visits = sorted({ref.dataId['visit'] for ref in refs})
print(len(all_visits), 'visits with single_visit_star',f" limit = {LIMIT}")

visit_info = None
try:
    records = run_butler.registry.queryDimensionRecords(
        'visit', where='instrument = inst AND visit IN (vs)', bind={'inst': 'LSSTCam', 'vs': all_visits})
    visit_info = pd.DataFrame([{
        'visit': rec.id,
        'band': rec.physical_filter.split('_')[0],
        'exp_time': getattr(rec, 'exposure_time', None),
        'zenith_angle': getattr(rec, 'zenith_angle', None),
        'day_obs': getattr(rec, 'day_obs', None),
        'observation_type': getattr(rec, 'observation_type', None),
        'observation_reason': getattr(rec, 'observation_reason', None),
        'science_program': getattr(rec, 'science_program', None),
        'target_name': getattr(rec, 'target_name', None),
    } for rec in records]).sort_values('visit').reset_index(drop=True)
    print(visit_info.groupby('band').size().rename('n_visits'))
    print(visit_info.head(10))
except Exception as exc:
    print('could not query the visit records:', exc)

visits_to_run = all_visits
if visit_info is not None and SAMPLE_PER_BAND is not None:
    chosen = []
    for band, g in visit_info.groupby('band'):
        vs = g['visit'].tolist()
        step = max(1, len(vs) // SAMPLE_PER_BAND)
        chosen += vs[::step][:SAMPLE_PER_BAND]
    visits_to_run = sorted(chosen)
if MAX_VISITS is not None:
    visits_to_run = visits_to_run[:MAX_VISITS]
if len(visits_to_run) > 500:
    print('visits processed in this run:', len(visits_to_run), visits_to_run[:500])
else:
    print('visits processed in this run:', len(visits_to_run), visits_to_run)

## 2. Per-visit processing

The Monster is read shard by shard and cached, since the visits overlap the same sky area.

In [ ]:
shard_cache = {}
dropped_coord = {}  # visit -> (rows dropped for non-finite ra/dec, total rows)
dropped_by_detector = {}  # visit -> {detector: rows dropped for non-finite ra/dec}


def monster_for_footprint(ra_c, dec_c, radius):
    """
    Retrieve the Monster catalog stars in the shards falling inside the cone specified in the arguments 
    """
    # Monster rows of all HTM7 shards touching the cone (shards are cached)
    frames = []
    for htm_id in mon.htm_ids_for_cone(ra_c, dec_c, radius):
        if htm_id not in shard_cache:
            tab = mon.load_monster_shards(run_butler, [htm_id], dataset=MONSTER_DATASET,
                                          collections=MONSTER_COLLECTIONS, verbose=False)
            if tab is None:
                shard_cache[htm_id] = None
            else:
                frame = mon.as_dataframe(tab)
                frame['ra_deg'] = mon.to_degrees(tab['coord_ra'])
                frame['dec_deg'] = mon.to_degrees(tab['coord_dec'])
                shard_cache[htm_id] = frame
        if shard_cache[htm_id] is not None:
            frames.append(shard_cache[htm_id])
    if not frames:
        raise RuntimeError(f'no Monster shard found for footprint centre ({ra_c:.3f}, {dec_c:.3f}) deg, radius {radius:.2f} deg')
    return pd.concat(frames, ignore_index=True)


def process_visit(visit):
    """
    match LSST stars defined in single_visit_star with monster catalog stars
    """
    src_df = mon.as_dataframe(run_butler.get('single_visit_star', visit=visit, instrument='LSSTCam'))
    n_all = len(src_df)
    finite = np.isfinite(src_df['ra']) & np.isfinite(src_df['dec'])
    dropped_by_detector[visit] = src_df.loc[~finite, 'detector'].value_counts().to_dict()
    src_df = src_df[finite].reset_index(drop=True)
    dropped_coord[visit] = (n_all - len(src_df), n_all)
    if len(src_df) == 0:
        raise RuntimeError(f'all {n_all} sources of visit {visit} have non-finite ra/dec')
    band = str(src_df['band'].iloc[0])

    ra_c = float(src_df['ra'].median())
    dec_c = float(src_df['dec'].median())
    
    pts = SkyCoord(src_df['ra'].to_numpy() * u.deg, src_df['dec'].to_numpy() * u.deg)
    radius = float(SkyCoord(ra_c * u.deg, dec_c * u.deg).separation(pts).deg.max()) + 0.05
    mon_df = monster_for_footprint(ra_c, dec_c, radius)

    # reference selection of calibrateImage (colour cut on ComCam g-r) + valid flux in this band
    mag = {b: mon.flux_njy_to_abmag(mon_df[f'{REF}_{b}_flux']) for b in 'griz'}
    gr = mag['g'] - mag['r']
    flux = mon_df[f'{REF}_{band}_flux'].to_numpy()

    #cut on color g-r
    sel = np.isfinite(gr) & (gr >= 0.4) & (gr <= 2.0) & (flux > 0)
    
    ref = mon_df[sel].reset_index(drop=True)
    gi = (mag['g'] - mag['i'])[sel]
    gaia_out = ((mon.flux_njy_to_abmag(ref['phot_g_mean_flux']) - mag['r'][sel]) > 1.0)

    #calculate crossmatching
    idx, sep, ok = mon.crossmatch(src_df['ra'], src_df['dec'], ref['ra_deg'], ref['dec_deg'],
                                  max_sep_arcsec = MAX_SEP_ARCSEC)
    
    cols = [c for c in ('detector', 'psfFlux', 'psfFluxErr', 'x', 'y') if c in src_df.columns]
    match = src_df.loc[ok, cols].reset_index(drop=True)
    match['visit'] = visit
    match['band'] = band
    match['sep_arcsec'] = sep[ok]
    match['m_src'] = mon.flux_njy_to_abmag(match['psfFlux'])
    match['m_ref'] = mon.flux_njy_to_abmag(ref[f'{REF}_{band}_flux'].to_numpy()[idx[ok]])
    match['gr_ref'] = gr[sel][idx[ok]]
    match['gi_ref'] = gi[idx[ok]]
    match['flag'] = ref[f'{REF}_{band}_source_flag'].to_numpy()[idx[ok]]
    match['monster_id'] = ref['id'].to_numpy()[idx[ok]]
    match['gaia_outlier'] = gaia_out[idx[ok]]
    # compute difference in mag
    match['dmag'] = match['m_src'] - match['m_ref']
    return match[np.isfinite(match['dmag'])].reset_index(drop=True)

### Matching stars from the visits with monster stars 

In [ ]:
results = []
failures = []
# loop on visit to process_visit
for n, visit in enumerate(visits_to_run):
    try:
        results.append(process_visit(visit))
    except Exception as exc:
        failures.append((visit, repr(exc)))
    if (n + 1) % 100 == 0 or n + 1 == len(visits_to_run):
        print(f'{n + 1}/{len(visits_to_run)} visits processed, {len(failures)} failures, {len(shard_cache)} shards cached')

print('first failures:', failures[:5])
allm = pd.concat(results, ignore_index=True)
allm.to_csv(DATA_DIR / 'matches_all.csv', index=False)
print("results for matching single_visit_star with monster stars in each band")
print(allm.shape)
print(allm.groupby('band')['visit'].nunique().rename('n_visits'))
display(allm.head())

## 3. Sensor type of each detector

In [ ]:
from lsst.obs.lsst import LsstCam

camera = LsstCam().getCamera()
sensor_type = {}
for det_id in sorted(allm['detector'].unique()):
    det = camera[int(det_id)]
    sensor_type[int(det_id)] = (det.getName(), det.getPhysicalType())
allm['sensor_type'] = allm['detector'].map(lambda d: sensor_type[int(d)][1])
print(pd.DataFrame(sensor_type, index=['name', 'type']).T)

## 4. Summaries

In [ ]:
core = allm[(allm['m_ref'] >= MAG_RANGE[0]) & (allm['m_ref'] <= MAG_RANGE[1])]

rows = []
for (visit, band, det), s in core.groupby(['visit', 'band', 'detector']):
    if len(s) >= MIN_STARS:
        med, sig = robust_stats(s['dmag'])
        rows.append({'visit': visit, 'band': band, 'detector': det, 'sensor_type': s['sensor_type'].iloc[0],
                     'n': len(s), 'median_dmag': med, 'sigma_mad': sig})
per_det = pd.DataFrame(rows)
per_det.to_csv(DATA_DIR / 'summary_per_detector.csv', index=False)
print(per_det.groupby(['band', 'sensor_type'])['median_dmag'].describe().round(4))

In [ ]:
# per-visit summary for ALL processed visits, with the observing conditions
rows = []
for (visit, band), s in core.groupby(['visit', 'band']):
    med, sig = robust_stats(s['dmag'])
    rows.append({'visit': visit, 'band': band, 'n': len(s), 'median_dmag': med, 'sigma_mad': sig,
                 'frac_abs_gt_0p1': float((s['dmag'].abs() > 0.1).mean())})
per_visit_summary = pd.DataFrame(rows)
if visit_info is not None:
    per_visit_summary = per_visit_summary.merge(
        visit_info[['visit', 'exp_time', 'zenith_angle', 'day_obs', 'observation_type', 'observation_reason',
                    'science_program', 'target_name']], on='visit', how='left')
per_visit_summary.to_csv(DATA_DIR / 'summary_per_visit.csv', index=False)

with pd.option_context('display.max_rows',100):
    print(per_visit_summary.round(4).to_string(index=False))
#display((per_visit_summary.round(4).head())

print('visits with non-finite coordinates (rows dropped, total rows):',
      {k: v for k, v in dropped_coord.items() if v[0] > 0})
print('detectors with non-finite coordinates, per visit:',
      {k: v for k, v in dropped_by_detector.items() if v})

In [ ]:
# colour term of the bright stars, flag 8 excluded: dmag = slope * (g - i) + offset
# least-squares fit (sensitive to outliers) and Theil-Sen fit (robust, with 95% interval)
from scipy.stats import theilslopes
rows = []
for (visit, band), s in core.groupby(['visit', 'band']):
    b = s[(s['m_ref'] < BRIGHT_MAX) & (s['flag'] != 8)]
    if len(b) >= 30 and b['gi_ref'].std() > 0.3:
        (slope, offset), cov = np.polyfit(b['gi_ref'], b['dmag'], 1, cov=True)
        ts_slope, ts_offset, ts_lo, ts_hi = theilslopes(b['dmag'], b['gi_ref'])
        med, sig = robust_stats(b['dmag'])
        rows.append({'visit': visit, 'band': band, 'n': len(b),
                     'slope_ls': slope, 'slope_ls_err': float(np.sqrt(cov[0, 0])),
                     'slope_ts': ts_slope, 'ts_lo': ts_lo, 'ts_hi': ts_hi, 'offset_ts': ts_offset,
                     'median_dmag': med, 'sigma_mad': sig,
                     'frac_abs_gt_0p1': float((b['dmag'].abs() > 0.1).mean())})
slopes = pd.DataFrame(rows)
slopes.to_csv(DATA_DIR / 'summary_colour_term.csv', index=False)

with pd.option_context('display.max_rows',100):
    print(slopes.round(4).to_string(index=False))
print(slopes.groupby('band')['slope_ts'].agg(['median', 'std', 'count']).round(4))

In [ ]:
# per-visit diagnostics: are the large scatter and offsets of some visits due to outliers?
print(slopes[['visit', 'band', 'n', 'median_dmag', 'sigma_mad', 'frac_abs_gt_0p1']].round(4).to_string(index=False))

visits_in = sorted(allm['visit'].unique())
ncol = 3
nrow = int(np.ceil(len(visits_in) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 4 * nrow), squeeze=False, sharex=True, sharey=True)
for ax, visit in zip(axes.ravel(), visits_in):
    s = allm[allm['visit'] == visit]
    band_v = s['band'].iloc[0]
    ax.plot(s['m_ref'], s['dmag'], '.', ms=3, alpha=0.5)
    ax.axhline(0, color='k', lw=1)
    ax.set_ylim(-0.5, 0.5)
    ax.set_title(f'{visit} ({band_v}), N={len(s)}')
    ax.set_xlabel('Monster mag')
    ax.set_ylabel('psfFlux mag - Monster mag')
savefig(fig, 'dmag_vs_mag_per_visit')

In [ ]:
# pooled colour term per band and sensor type, using only visits with a clean calibration
from scipy.stats import theilslopes

GOOD_SIGMA_MAD = 0.03
good_visits = per_visit_summary[per_visit_summary['sigma_mad'] < GOOD_SIGMA_MAD]['visit']
cg = core[core['visit'].isin(good_visits) & (core['m_ref'] < BRIGHT_MAX) & (core['flag'] != 8)]

groups = [((b, st), s) for (b, st), s in cg.groupby(['band', 'sensor_type'])]
groups += [((b, 'all'), s) for b, s in cg.groupby('band')]
rows = []
for (band, stype), s in groups:
    if len(s) >= 30 and s['gi_ref'].std() > 0.3:
        if len(s) > 3000:  # Theil-Sen is quadratic in the number of points
            s = s.sample(3000, random_state=0)
        ts_slope, ts_offset, ts_lo, ts_hi = theilslopes(s['dmag'], s['gi_ref'])
        rows.append({'band': band, 'sensor_type': stype, 'n_visits': s['visit'].nunique(), 'n_stars': len(s),
                     'slope_ts': ts_slope, 'ts_lo': ts_lo, 'ts_hi': ts_hi, 'offset_ts': ts_offset})
pooled = pd.DataFrame(rows).sort_values(['band', 'sensor_type'])
pooled.to_csv(DATA_DIR / 'summary_colour_term_pooled.csv', index=False)

with pd.option_context('display.max_rows',100):
    print(pooled.round(4).to_string(index=False))

In [ ]:
# is the scatter explained by the photon noise of psfFlux?  compare sigma_MAD(dmag) with the median psfFlux error
allm['sigma_mag'] = 1.0857 * allm['psfFluxErr'] / allm['psfFlux']
# u-band visits are all above the clean threshold: keep them to compare their scatter with the noise
clean = allm[(allm['visit'].isin(good_visits) | (allm['band'] == 'u')) & (allm['flag'] != 8)]
bins = np.arange(14.0, 22.5, 1.0)
rows = []
for band, sb in clean.groupby('band'):
    for lo, hi in zip(bins[:-1], bins[1:]):
        s = sb[(sb['m_ref'] >= lo) & (sb['m_ref'] < hi)]
        if len(s) >= 15:
            _, sig = robust_stats(s['dmag'])
            rows.append({'band': band, 'm_ref_lo': lo, 'n': len(s), 'sigma_mad_dmag': sig,
                         'median_sigma_psfFlux': s['sigma_mag'].median()})
noise_table = pd.DataFrame(rows)
noise_table.to_csv(DATA_DIR / 'summary_noise_vs_mag.csv', index=False)

with pd.option_context('display.max_rows',100):
    print(noise_table.round(4).to_string(index=False))

In [ ]:
# u band: where does the large scatter come from?  scatter by Monster source_flag and by Monster g - r colour
u_stars = allm[(allm['band'] == 'u') & (allm['m_ref'] >= MAG_RANGE[0]) & (allm['m_ref'] <= MAG_RANGE[1])]
print('by Monster flag:')
rows = []
for flag, s in u_stars.groupby('flag'):
    if len(s) >= 10:
        med, sig = robust_stats(s['dmag'])
        rows.append({'flag': int(flag), 'n': len(s), 'median_dmag': med, 'sigma_mad': sig,
                     'median_m_ref': s['m_ref'].median()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

print('by Monster g - r colour:')
rows = []
for lo, hi in ((0.4, 0.7), (0.7, 1.0), (1.0, 1.3), (1.3, 2.0)):
    s = u_stars[(u_stars['gr_ref'] >= lo) & (u_stars['gr_ref'] < hi)]
    if len(s) >= 10:
        med, sig = robust_stats(s['dmag'])
        rows.append({'g-r': f'{lo}-{hi}', 'n': len(s), 'median_dmag': med, 'sigma_mad': sig})

with pd.option_context('display.max_rows',100):
    print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
# repeatability: the same Monster star seen in two visits of the same band.
# If the Monster error dominates the scatter of dmag, dmag is the same in both visits (the difference is small);
# if the step-1 photometry dominates, the difference carries the scatter.
rep = allm[(allm['m_ref'] >= MAG_RANGE[0]) & (allm['m_ref'] <= MAG_RANGE[1]) & (allm['flag'] != 8)]
rows = []
for band, sb in rep.groupby('band'):
    piv = sb.pivot_table(index='monster_id', columns='visit', values='dmag', aggfunc='median')
    vis = piv.columns.tolist()
    for i in range(len(vis)):
        for j in range(i + 1, len(vis)):
            both = piv[[vis[i], vis[j]]].dropna()
            if len(both) >= 10:
                sig_tot = 0.5 * (robust_stats(both[vis[i]])[1] + robust_stats(both[vis[j]])[1])
                sig_step1 = robust_stats(both[vis[i]] - both[vis[j]])[1] / np.sqrt(2.0)
                rows.append({'band': band, 'visit_a': vis[i], 'visit_b': vis[j], 'n_common': len(both),
                             'sigma_total': sig_tot, 'sigma_step1_est': sig_step1,
                             'sigma_monster_est': float(np.sqrt(max(sig_tot ** 2 - sig_step1 ** 2, 0.0)))})
repeat = pd.DataFrame(rows)
if len(repeat):
    repeat.to_csv(DATA_DIR / 'summary_repeatability.csv', index=False)
    with pd.option_context('display.max_rows',100):
        print(repeat.round(4).to_string(index=False))
else:
    print('no Monster star in common between two visits of the same band (more visits needed)')

In [ ]:
# do the visits with a large scatter cluster in time (nights)?  (u band excluded, at least 100 matched stars)
pvs = per_visit_summary[per_visit_summary['band'] != 'u'].copy()
pvs['bad'] = (pvs['sigma_mad'] > GOOD_SIGMA_MAD) & (pvs['n'] >= 100)
if 'day_obs' in pvs.columns:
    by_night = pvs.groupby('day_obs').agg(n_visits=('visit', 'size'), n_bad=('bad', 'sum'),
                                          median_zenith=('zenith_angle', 'median'))
    by_night['frac_bad'] = by_night['n_bad'] / by_night['n_visits']
    by_night.to_csv(DATA_DIR / 'summary_bad_visits_by_night.csv')
    print(by_night[by_night['n_bad'] > 0].sort_values('n_bad', ascending=False).round(3).to_string())
    print('nights with at least one bad visit:', int((by_night['n_bad'] > 0).sum()), 'of', len(by_night))
    print('bad visits:', int(pvs['bad'].sum()), 'of', len(pvs))

In [ ]:
# repeatability before and after removing the colour term (Theil-Sen slope per band of the pooled fit, rows 'all')
# the pair loop is quadratic in the number of visits: use a random subset of the visits with the most stars
REP_VISITS_PER_BAND = 20
slope_by_band = pooled[pooled['sensor_type'] == 'all'].set_index('band')['slope_ts'].to_dict()
rng = np.random.default_rng(0)
rep2 = allm[(allm['m_ref'] >= MAG_RANGE[0]) & (allm['m_ref'] <= MAG_RANGE[1]) & (allm['flag'] != 8)
            & allm['visit'].isin(good_visits)].copy()
rep2['dmag_corr'] = rep2['dmag'] - rep2['band'].map(slope_by_band).fillna(0.0) * rep2['gi_ref']

rows = []
for band, sb in rep2.groupby('band'):
    candidates = sb.groupby('visit').size().sort_values(ascending=False).index[:200]
    picked = list(rng.choice(candidates, size=min(REP_VISITS_PER_BAND, len(candidates)), replace=False))
    sb = sb[sb['visit'].isin(picked)]
    for col in ('dmag', 'dmag_corr'):
        piv = sb.pivot_table(index='monster_id', columns='visit', values=col, aggfunc='median')
        vis = piv.columns.tolist()
        for i in range(len(vis)):
            for j in range(i + 1, len(vis)):
                both = piv[[vis[i], vis[j]]].dropna()
                if len(both) >= 30:
                    sig_tot = 0.5 * (robust_stats(both[vis[i]])[1] + robust_stats(both[vis[j]])[1])
                    sig_step1 = robust_stats(both[vis[i]] - both[vis[j]])[1] / np.sqrt(2.0)
                    rows.append({'band': band, 'quantity': col, 'n_common': len(both), 'sigma_total': sig_tot,
                                 'sigma_step1_est': sig_step1,
                                 'sigma_common_est': float(np.sqrt(max(sig_tot ** 2 - sig_step1 ** 2, 0.0)))})
rep_tab = pd.DataFrame(rows)
rep_tab.to_csv(DATA_DIR / 'summary_repeatability_colour_corrected.csv', index=False)
if len(rep_tab):
    out = rep_tab.groupby(['band', 'quantity']).apply(lambda s: pd.Series({
        'pairs': len(s), 'stars': s['n_common'].sum(),
        'sigma_total': np.average(s['sigma_total'], weights=s['n_common']),
        'sigma_step1': np.average(s['sigma_step1_est'], weights=s['n_common']),
        'sigma_common': np.average(s['sigma_common_est'], weights=s['n_common'])}))
    print(out.round(4).to_string())
else:
    print('no pair of visits with at least 30 common stars')

In [ ]:
# u band: scatter by Monster flag and magnitude bin (all u visits)
u_all = allm[allm['band'] == 'u']
rows = []
for flag, sf in u_all.groupby('flag'):
    for lo, hi in ((14, 16), (16, 17), (17, 18), (18, 19), (19, 20)):
        s = sf[(sf['m_ref'] >= lo) & (sf['m_ref'] < hi)]
        if len(s) >= 20:
            med, sig = robust_stats(s['dmag'])
            rows.append({'flag': int(flag), 'm_ref': f'{lo}-{hi}', 'n': len(s), 'median_dmag': med,
                         'sigma_mad': sig, 'median_sigma_psfFlux': float(s['sigma_mag'].median())})
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
# position dependence of dmag inside the detectors of the visits whose PSF model failed
bad_nonu = per_visit_summary[(per_visit_summary['sigma_mad'] > GOOD_SIGMA_MAD)
                             & (per_visit_summary['band'] != 'u')]['visit'].tolist()
if 'x' in allm.columns and bad_nonu:
    fig, axes = plt.subplots(len(bad_nonu), 2, figsize=(14, 4.5 * len(bad_nonu)), squeeze=False)
    for row, visit in zip(axes, bad_nonu):
        s = allm[(allm['visit'] == visit) & (allm['m_ref'] >= MAG_RANGE[0]) & (allm['m_ref'] <= MAG_RANGE[1])]
        for ax, col in zip(row, ('x', 'y')):
            for det, sd in s.groupby('detector'):
                ax.plot(sd[col], sd['dmag'], '.', ms=4, alpha=0.6, label=str(det))
            ax.axhline(0, color='k', lw=1)
            ax.set_ylim(-0.3, 0.6)
            ax.set_xlabel(f'{col} (pixels)')
            ax.set_ylabel('psfFlux mag - Monster mag')
            ax.set_title(f'visit {visit}')
    axes[0][0].legend(title='detector', ncol=2)
    savefig(fig, 'bad_visits_dmag_vs_position')
else:
    print('x, y not available: rerun the processing loop (cell 6) after this change')

## 5. Figures

In [ ]:
order = {v: i for i, v in enumerate(sorted(allm['visit'].unique()))}
per_visit = core.groupby(['visit', 'band'])['dmag'].median().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for band, s in per_visit.groupby('band'):
    s = s.sort_values('visit')
    axes[0].plot(s['visit'].map(order), s['dmag'], 'o-', label=band)
axes[0].axhline(0, color='k', lw=1)
axes[0].set_xlabel('visit index (sorted)')
axes[0].set_ylabel('median(psfFlux mag - Monster mag)')
axes[0].legend()

for band, s in slopes.groupby('band'):
    s = s.sort_values('visit')
    axes[1].errorbar(s['visit'].map(order), s['slope_ts'],
                     yerr=[s['slope_ts'] - s['ts_lo'], s['ts_hi'] - s['slope_ts']], fmt='o', label=band)
axes[1].axhline(0, color='k', lw=1)
axes[1].set_xlabel('visit index (sorted)')
axes[1].set_ylabel('colour term (Theil-Sen): slope of dmag vs g - i')
axes[1].legend()
savefig(fig, 'per_visit_offsets_and_colour_term')

In [ ]:
# median offset per detector and band, labelled with the sensor type
pivot = per_det.groupby(['detector', 'band'])['median_dmag'].median().unstack()
labels = [f'{d} ({sensor_type[int(d)][1]})' for d in pivot.index]

fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot.bar(ax=ax)
ax.set_xticklabels(labels, rotation=90)
ax.axhline(0, color='k', lw=1)
ax.set_ylabel('median(psfFlux mag - Monster mag)')
savefig(fig, 'per_detector_offsets')
print(pivot.round(4))